In [ ]:
"""[Kaggle L4] TRIAGE del TTT en el 4B: quien lo necesita, a quien mata, cuando converge.

Contexto (22-ago): nottt = 9,58 vs campeon 30,14 -> el TTT vale +20,6 pt (2/3 del score) por
34,8% del reloj. La linea nueva es AMPLIFICARLO, y este kernel produce los datos para las tres
ramas a la vez:

1. TRIAGE (paso 1 del usuario): por tarea, resultado con y sin TTT + las tres observables
   candidatas a disparador (confianza base, verificador paleta+tamano, celdas dudosas). Las
   politicas selectivas se evaluan OFFLINE contra esto. Incluye el grupo de riesgo: las 3
   tareas que la base resuelve (bf45cf4b, 38007db0, 6e453dd6) -- mide si el TTT del 4B MATA
   tareas resueltas (fenomeno b2bc3ffd, nunca medido en el 4B).
2. CONVERGENCIA (gate del paso 2, meta-aprendizaje): NLL de la verdad en los pasos 12/24/36/48
   -- si la mayoria converge a mitad de camino, la ganancia rapida es reasignar pasos, no
   meta-entrenar.
3. PICOS: vectores por celda pre/post TTT -- ver los picos de 2-10 nats aplastarse (o no), la
   materia prima del CE ponderado.
4. POOLS VOLCADOS: rejillas de todos los candidatos (base y TTT) -> tasa de caza del
   verificador (FP=0 ya medido sobre 120 verdades) sobre candidatos INCORRECTOS, offline.

TTT fiel a produccion: LoRA r=256 + rsLoRA (alpha 32 -> escala 2,0), 7 proyecciones +
embed_tokens + lm_head, augmentaciones d4 + permutacion de color + orden de ejemplos,
mascara a los spans de salida. Decode de vista identidad (limitacion anotada: produccion
decodifica 16 vistas y agrega).
"""
import glob
import json
import os
import random
import time

import numpy as np
import torch
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import AutoModelForCausalLM

T00 = time.time()
_SDPA = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]
IM_START, USER, ASSIST, NL, EOS = 14, 11, 12, 10, 15
ARC_TOKENS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15]
_TOKID = torch.tensor(ARC_TOKENS, dtype=torch.long, device="cuda:0")
THR = -float(np.log(0.2))
N_AUG = 12
EPOCAS_TTT = 4                 # 12 vistas x 4 = 48 pasos
CHECKPOINTS = (12, 24, 36, 48)
LR = 5e-5
MAX_SEQ = 3500

_COMP = None
for c in glob.glob("/kaggle/input/**/arc-agi_evaluation_challenges.json", recursive=True):
    _COMP = os.path.dirname(c)
    break
assert _COMP, "no se encontro la data de la competencia"
MDIR = None
for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
    d = os.path.dirname(c)
    if glob.glob(os.path.join(d, "*.safetensors")):
        MDIR = d
        break
assert MDIR, "no se encontro el modelo adjunto"
print(f"[setup] comp={_COMP} modelo={MDIR}", flush=True)

CHAL = json.load(open(f"{_COMP}/arc-agi_evaluation_challenges.json"))
SOLS = json.load(open(f"{_COMP}/arc-agi_evaluation_solutions.json"))

# 3 que la base resuelve (grupo de riesgo de muerte por TTT) + 17 por coste
KEYS = ["bf45cf4b", "38007db0", "6e453dd6",
        "e8686506", "20270e3b", "7b5033c1", "b0039139", "78332cb0", "28a6681f", "d35bdbdc",
        "dbff022c", "4c3d4a41", "446ef5d2", "dd6b8c4b", "3dc255db", "16de56c4", "97d7923e",
        "2b83f449", "faa9f03d", "eee78d87"]
GATE = "38007db0"


def grid_tokens(g):
    out = []
    for i, row in enumerate(g):
        out.extend(int(c) for c in row)
        if i < len(g) - 1:
            out.append(NL)
    return out


def pair_tokens_mask(inp, out):
    t = [IM_START, USER, NL] + grid_tokens(inp) + [EOS]
    m = [0] * len(t)
    head = [IM_START, ASSIST, NL]
    body = grid_tokens(out) + [EOS]
    return t + head + body, m + [0] * len(head) + [1] * len(body)


def query_tokens(inp):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + [IM_START, ASSIST, NL]


# --- augmentaciones (d4 + permutacion de color + orden de ejemplos) ---

def d4(g, k):
    a = np.array(g)
    if k >= 4:
        a = a.T
    for _ in range(k % 4):
        a = np.rot90(a)
    return a.tolist()


def aplicar(g, k, perm):
    return [[perm[c] for c in fila] for fila in d4(g, k)]


def build_ttt(task, seed):
    rng = random.Random(seed)
    pares = [(p["input"], p["output"]) for p in task["train"]]
    seqs = []
    for v in range(N_AUG):
        if v == 0:
            k, perm, orden = 0, list(range(10)), list(range(len(pares)))
        else:
            k = rng.randrange(8)
            perm = list(range(10))
            rng.shuffle(perm)
            orden = list(range(len(pares)))
            rng.shuffle(orden)
        toks, mask = [], []
        for idx in orden:
            i, o = pares[idx]
            t, m = pair_tokens_mask(aplicar(i, k, perm), aplicar(o, k, perm))
            toks += t
            mask += m
        if len(toks) <= MAX_SEQ:
            seqs.append((toks, mask))
    return seqs


base = AutoModelForCausalLM.from_pretrained(MDIR, torch_dtype=torch.bfloat16,
                                            device_map="cuda:0", attn_implementation="sdpa")
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, \
    set_peft_model_state_dict

LCFG = LoraConfig(r=256, lora_alpha=32, use_rslora=True, lora_dropout=0.0, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj",
                                  "embed_tokens", "lm_head"])
MODEL = get_peft_model(base, LCFG)
for _, p_ in MODEL.named_parameters():
    if p_.requires_grad:
        p_.data = p_.data.float()
MODEL.enable_input_require_grads()
MODEL.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
INIT = {k: v.detach().clone() for k, v in get_peft_model_state_dict(MODEL).items()}
try:
    import bitsandbytes as bnb
    OPT_CLS = bnb.optim.AdamW8bit
except Exception:                                            # noqa: BLE001
    OPT_CLS = torch.optim.AdamW
print(f"[setup] 4B + LoRA r=256 (rsLoRA, embed+head) | opt={OPT_CLS.__name__} | "
      f"mem={torch.cuda.memory_allocated()/1e9:.1f}GB ({time.time()-T00:.0f}s)", flush=True)


@torch.no_grad()
def prefill(toks, chunk=512):
    cache, out = None, None
    for i in range(0, len(toks), chunk):
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=torch.tensor([toks[i:i + chunk]], device="cuda:0",
                                               dtype=torch.long),
                        past_key_values=cache, use_cache=True, return_dict=True)
        cache = out.past_key_values
    return out.logits[:, -1], cache


@torch.no_grad()
def nll_vector(prompt, ans):
    last, cache = prefill(prompt)
    parts = [last.float()]
    if len(ans) > 1:
        with sdpa_kernel(_SDPA):
            o = MODEL(input_ids=torch.tensor([ans[:-1]], device="cuda:0", dtype=torch.long),
                      past_key_values=cache, use_cache=True, return_dict=True)
        parts.append(o.logits[0].float())
    lg = torch.cat([x.view(-1, x.shape[-1]) for x in parts], 0)
    v = -torch.log_softmax(lg, -1).gather(
        1, torch.tensor(ans, device="cuda:0").unsqueeze(1)).squeeze(1)
    out = [round(x, 5) for x in v.tolist()]
    del parts, lg, cache
    torch.cuda.empty_cache()
    return out


def toks_to_grid(toks):
    rows, row = [], []
    for t in toks:
        if t == NL:
            rows.append(row)
            row = []
        elif 0 <= t <= 9:
            row.append(t)
        else:
            return None
    if row:
        rows.append(row)
    if not rows or not rows[0] or any(len(r) != len(rows[0]) for r in rows):
        return None
    return rows


@torch.no_grad()
def dfs(prompt, depth, budget=45):
    t0 = time.time()
    last, cache0 = prefill(prompt)
    leaves, path = [], []

    def rec(logits, score, cache, left):
        if time.time() - t0 > budget or left <= 1:
            return
        lf = logits.float()
        nll = (score + torch.logsumexp(lf, dim=-1, keepdim=True)
               - lf.index_select(-1, _TOKID)).cpu()
        cands = []
        for j, t in enumerate(ARC_TOKENS):
            sc = nll[0, j].item()
            if sc < THR:
                if t == EOS:
                    leaves.append((sc, list(path)))
                else:
                    cands.append((sc, t))
        for sc, t in sorted(cands):
            if time.time() - t0 > budget:
                return
            with sdpa_kernel(_SDPA):
                o = MODEL(input_ids=torch.tensor([[t]], device="cuda:0", dtype=torch.long),
                          past_key_values=cache, use_cache=True, return_dict=True)
            path.append(t)
            rec(o.logits[:, -1], sc, o.past_key_values, left - 1)
            path.pop()

    rec(last, 0.0, cache0, depth)
    return leaves, time.time() - t0


def make_prompt(k):
    t = CHAL[k]
    p = []
    for x in t["train"]:
        tt, _ = pair_tokens_mask(x["input"], x["output"])
        p += tt
    p += query_tokens(t["test"][0]["input"])
    return p


def pool_dump(hojas):
    out = []
    for sc, p in sorted(hojas, key=lambda x: x[0])[:8]:
        g = toks_to_grid(p)
        if g is not None:
            out.append({"score": round(sc, 4), "grid": g})
    return out


def modo_eval():
    MODEL.eval()
    MODEL.config.use_cache = True


def modo_train():
    MODEL.config.use_cache = False
    MODEL.train()


# GATE (con adapter en INIT = base exacta: los B de LoRA nacen en cero)
set_peft_model_state_dict(MODEL, {k: v.clone() for k, v in INIT.items()})
modo_eval()
_v = nll_vector(make_prompt(GATE), grid_tokens(SOLS[GATE][0]) + [EOS])
print(f"[gate] {GATE}: NLL {sum(_v):.4f}", flush=True)
assert sum(_v) < 1.0, f"gate FALLA ({sum(_v):.3f}): modelo mal cargado"

RES = {"tareas": [], "n_aug": N_AUG, "pasos": N_AUG * EPOCAS_TTT, "lr": LR}
for i, k in enumerate(KEYS):
    task = CHAL[k]
    gt = SOLS[k][0]
    prompt = make_prompt(k)
    gt_toks = grid_tokens(gt) + [EOS]
    depth = min(931, len(gt_toks) + 40)

    # --- brazo BASE ---
    set_peft_model_state_dict(MODEL, {kk: v.clone() for kk, v in INIT.items()})
    modo_eval()
    v_pre = nll_vector(prompt, gt_toks)
    hojas0, s0 = dfs(prompt, depth)
    pool0 = pool_dump(hojas0)
    ok0 = bool(pool0 and pool0[0]["grid"] == gt)
    alguno0 = bool(any(c["grid"] == gt for c in pool0))

    # --- TTT ---
    seqs = build_ttt(task, seed=1000 + i)
    modo_train()
    opt = OPT_CLS([p for p in MODEL.parameters() if p.requires_grad], lr=LR)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=len(seqs) * EPOCAS_TTT)
    curva, losses, paso = [], [], 0
    t0 = time.time()
    try:
        for ep in range(EPOCAS_TTT):
            for toks, mask in seqs:
                ids = torch.tensor([toks], device="cuda:0", dtype=torch.long)
                lab = torch.tensor([[t if m else -100 for t, m in zip(toks, mask)]],
                                   device="cuda:0", dtype=torch.long)
                with sdpa_kernel(_SDPA):
                    out = MODEL(input_ids=ids, labels=lab)
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [p for p in MODEL.parameters() if p.requires_grad], 1.0)
                opt.step()
                sch.step()
                opt.zero_grad(set_to_none=True)
                losses.append(out.loss.item())
                paso += 1
                if paso in CHECKPOINTS:
                    MODEL.zero_grad(set_to_none=True)
                    modo_eval()
                    vv = nll_vector(prompt, gt_toks)
                    curva.append({"paso": paso, "gt_nll": round(sum(vv), 4),
                                  "max_celda": round(max(vv), 4),
                                  "celdas_2n": sum(1 for x in vv if x > 2.0)})
                    modo_train()
    except torch.cuda.OutOfMemoryError:
        print(f"  [{k}] OOM en paso {paso}", flush=True)
    ttt_s = time.time() - t0
    del opt, sch
    MODEL.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()

    # --- brazo TTT ---
    modo_eval()
    v_post = nll_vector(prompt, gt_toks)
    hojas1, s1 = dfs(prompt, depth)
    pool1 = pool_dump(hojas1)
    ok1 = bool(pool1 and pool1[0]["grid"] == gt)
    alguno1 = bool(any(c["grid"] == gt for c in pool1))

    cat = ("mantiene" if ok0 and ok1 else "MATA" if ok0 and not ok1
           else "RESCATA" if ok1 and not ok0 else "ninguno")
    RES["tareas"].append({
        "task": k, "L": len(gt_toks), "categoria": cat,
        "base": {"ok": ok0, "gt_en_pool": alguno0, "gt_nll": round(sum(v_pre), 4),
                 "max_celda": round(max(v_pre), 4),
                 "celdas_2n": sum(1 for x in v_pre if x > 2.0),
                 "best_score": pool0[0]["score"] if pool0 else None,
                 "pool": pool0, "decode_s": round(s0, 1)},
        "ttt": {"ok": ok1, "gt_en_pool": alguno1, "gt_nll": round(sum(v_post), 4),
                "max_celda": round(max(v_post), 4),
                "celdas_2n": sum(1 for x in v_post if x > 2.0),
                "best_score": pool1[0]["score"] if pool1 else None,
                "pool": pool1, "decode_s": round(s1, 1), "ttt_s": round(ttt_s, 1),
                "loss_fin": round(float(np.mean(losses[-8:])), 5) if losses else None},
        "curva": curva, "v_pre": v_pre, "v_post": v_post})
    print(f"[{i+1}/{len(KEYS)}] {k}: {cat:8} | NLL {sum(v_pre):8.3f}->{sum(v_post):8.3f} | "
          f"celdas>2n {sum(1 for x in v_pre if x > 2.0)}->{sum(1 for x in v_post if x > 2.0)} | "
          f"pool {len(pool0)}->{len(pool1)} | ttt {ttt_s:.0f}s", flush=True)
    json.dump(RES, open("/kaggle/working/ttt_triage.json", "w"))
    torch.cuda.empty_cache()

t = RES["tareas"]
RES["resumen"] = {
    "n": len(t),
    "base_ok": sum(x["base"]["ok"] for x in t),
    "ttt_ok": sum(x["ttt"]["ok"] for x in t),
    "mata": sum(1 for x in t if x["categoria"] == "MATA"),
    "rescata": sum(1 for x in t if x["categoria"] == "RESCATA"),
    "celdas_2n_pre": sum(x["base"]["celdas_2n"] for x in t),
    "celdas_2n_post": sum(x["ttt"]["celdas_2n"] for x in t),
    "ttt_total_s": round(sum(x["ttt"]["ttt_s"] for x in t), 1),
}
json.dump(RES, open("/kaggle/working/ttt_triage.json", "w"))
print("RESULT_JSON:" + json.dumps(RES["resumen"]), flush=True)
print(f"[fin] {time.time()-T00:.0f}s", flush=True)
